# Unit 7 exercises: semantic text embeddings

Run the recsys data generators before you begin. Unit 3's lexical path called two books similar only when they shared surface **words**; these exercises match on **meaning** instead. Each vocabulary token carries a pretrained, dense **GloVe** vector (a point in a 100-dimensional semantic space where nearness reflects meaning), a book is the **mean** of its keyword tokens' vectors (L2-normalized), a reader is the aggregate of the books they have seen, and retrieval is **brute-force cosine** over the catalog. You first read the geometry directly (cosine nearest words, scoped to the catalog's own vocabulary), build a book and reader embedding by hand and confirm it against the shipped `book_embedding`, then register `SemanticEmbeddingRetrievalPath` and read its validation scoreboard.

Read it **honestly**. Measured on this data (seed 0, k=10, 500 val readers), semantic mean-pooling scores hit@10 ~0.102: well above the random floor (~0.012, about 8.5x) but **below** the lexical path (~0.158, about 65% of it), and far below the collaborative item-item CF (~0.252) and matrix-factorization (~0.276) paths. It is **not** a win. The reason is honest and specific: this catalog's latent-taste topics are arbitrary contiguous slices of a general word pool, so GloVe's meaning-geometry is uncorrelated-by-construction with the taste structure; only the genre banks carry GloVe-exploitable meaning. On a real catalog's prose, dense semantics would fare differently -- we teach the *method* honestly, not a staged win. Semantic's real value is **complementarity**: its top-10 overlap with lexical is only ~0.22, so it surfaces meaning-related books lexical misses, and the two Challenge measurements show that a weaker pooling can rank higher and that adding semantic to the Unit 6 blend lifts both hit *and* coverage.

Everything is measured on the **validation** split (`val`); each reader's **seen** set is their train-positive history. The `test` holdout stays sealed -- no cell here may touch it. Each answer is left for you to complete; no solutions live here.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PathRegistry,
    PopularityRetrievalPath,
    RandomRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    book_embedding,
    load_glove_subset,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
)
from bookrec.lexical import tokenize   # the library's tokenizer (what book_embedding uses internally)

SEED = 0
K = 10        # every scoreboard reports at the top-10 cutoff

# --- the real interaction log (the same substrate as Units 1-6) ----------------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)                      # 2000 books
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")  # {item_id: "space joined token bag"}

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])           # 60 readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# The committed, checksum-pinned GloVe subset: a (608, 100) matrix + a token->row index, loaded with
# numpy only (no gensim). One vocabulary word ('starfall') is in the vocabulary but GloVe lacked it:
# a zero row, listed in glove.missing, that drops out at pooling.
glove = load_glove_subset()

# Each reader's train-positive history is their `seen` (exclusion) set; their val positives are the
# relevance target. We grade on `val`; no cell here may read the sealed `test` split.
seen_by = defaultdict(set)
val_pos_by = defaultdict(set)
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_pos_by[row["reader_id"]].add(row["item_id"])


## Exercise 1 - Cosine nearest words (scoped to the catalog's vocabulary)

A GloVe vector turns a word into a point in a 100-dimensional semantic space, and **cosine similarity** between two such vectors measures how related the words are: `cos(a, b) = (a . b) / (|a| |b|)`, which is 1.0 for identical directions and near 0 for unrelated ones. Because every book embedding is built from these word vectors, the quickest way to *see* the space is to find a word's nearest neighbours.

For the query word below, compute the cosine similarity of its GloVe vector to **every** word in the subset's vocabulary and report the 5 nearest (skip the query word itself and any out-of-vocabulary zero row). Report the ordered list exactly as it comes out: for `"dragon"` the top 5 is `lantern` (~0.68), then `demon`, `sword`, `ghost`, `monster`. Most are clearly meaning-related, but note the lists are **noisy** -- these are only 100-dimensional vectors trained on Wikipedia co-occurrence, restricted to a 607-word field, so an odd neighbour (here `lantern`) can edge out the obvious one. Label your result honestly: these are the nearest among **the catalog's own 607-word vocabulary**, not the full-GloVe analogies a 400k-word model would give -- the committed subset is restricted on purpose. In two or three sentences, say what "distance = meaning" buys us that Unit 3's shared-token matching could never see.

In [ ]:
query_token = "dragon"   # try "ocean", "wizard", "castle" too -- all in the vocabulary

matrix = glove.matrix                       # (608, 100) float32; row i is glove.tokens[i]
norms = np.linalg.norm(matrix, axis=1)      # per-row length; the one OOV row has length 0
norms[norms == 0.0] = 1.0                    # avoid divide-by-zero on the zero (OOV) row
unit = matrix / norms[:, None]              # every row L2-normalized -> cosine is a dot product

query_row = glove.token_index[query_token]  # the query word's row in the matrix
cosines = None   # TODO: unit @ unit[query_row]  -- cosine of the query against every vocab word

# TODO: rank the vocabulary by `cosines` (descending) and take the 5 nearest TOKENS, skipping the
# query word's own row and any token in glove.missing (the OOV zero row).
#   order = np.argsort(-cosines)
#   nearest_tokens = []
#   for row in order:
#       token = glove.tokens[row]
#       if row == query_row or token in glove.missing:
#           continue
#       nearest_tokens.append(token)
#       if len(nearest_tokens) == 5:
#           break
nearest_tokens = None   # TODO: list[str] - the 5 nearest vocabulary words to query_token

query_excluded = None       # TODO: bool - query_token not in nearest_tokens
five_neighbours = None      # TODO: bool - len(nearest_tokens) == 5
nearest_label = "nearest among the catalog's 607-word vocabulary (not full-GloVe analogies)"
meaning_explanation = "TODO"  # 2-3 sentences: cosine distance in GloVe space reflects MEANING, so
                              # related words are near even with no shared letters -- something Unit
                              # 3's shared-token (bag-of-words) matching can never see

## Exercise 2 - Build a book and a reader embedding by hand

A book is a **bag of keyword tokens** (the same text Unit 3 indexed). Its **document embedding** is the simplest possible composition: the **mean** of its in-vocabulary tokens' GloVe vectors, then **L2-normalized** so every book is a unit vector and cosine is a plain dot product. Two kinds of token earn no vector: a made-up word absent from the subset vocabulary, for which `glove.vector(token)` returns `None`, and a vocabulary word GloVe itself lacked (`starfall`), for which `glove.vector(token)` returns an all-**zero row** (not `None`). Both drop out at pooling, and a book whose every token is missing must be a safe all-**zero** vector, never a NaN.

Build one book's embedding **by hand**: tokenize its keyword bag with `tokenize(...)` (what the library uses), take `glove.vector(token)` for each, **skip the tokens that come back `None`**, average the rest, and **L2-normalize only when the mean has nonzero length** (so an all-missing book returns zeros, not NaN). Confirm it matches the shipped `book_embedding(tokens, glove)` to floating-point precision, and confirm an all-out-of-vocabulary book comes back as the zero vector. Then aggregate a **reader embedding** the same way -- the mean of the embeddings of the books in one warm reader's `seen` set, L2-normalized -- which is exactly the query vector the retrieval path will score against the catalog. In two or three sentences, explain why L2-normalizing matters (so a book with many keywords cannot outscore a book with few purely by vector length).

In [ ]:
book_id = catalog_ids[0]
tokens = tokenize(keywords[book_id])        # this book's keyword bag as a token list (library tokenizer)

# --- by hand: mean of the in-vocabulary token vectors, then a ZERO-SAFE L2-normalize ---
token_vectors = None     # TODO: [glove.vector(t) for t in tokens if glove.vector(t) is not None]  (skip made-up words)
mean_vector = None       # TODO: np.mean(np.array(token_vectors), axis=0)
mean_norm = None         # TODO: float(np.linalg.norm(mean_vector))
my_embedding = None      # TODO: mean_vector / mean_norm if mean_norm != 0.0 else np.zeros(glove.dim)  (zero-safe)

library_embedding = None     # TODO: book_embedding(tokens, glove)
embeddings_match = None      # TODO: bool - np.allclose(my_embedding, library_embedding, atol=1e-5)
is_unit_length = None        # TODO: bool - np.isclose(np.linalg.norm(library_embedding), 1.0)

# A book whose EVERY token is out of vocabulary must be a safe zero vector, never NaN. 'starfall' is an
# in-vocabulary word GloVe lacked (a zero row); 'notaword' is absent from the subset (glove.vector -> None).
all_missing = book_embedding(["starfall", "notaword"], glove)
all_missing_is_zero = None   # TODO: bool - not np.any(all_missing)  (all-zero also rules out NaN: a NaN is truthy)

# --- a reader embedding: the mean of their seen books' embeddings, L2-normalized ---
reader_id = sorted(r for r in seen_by if r not in cold_readers)[0]   # the first warm reader
seen = seen_by[reader_id]
seen_embeddings = None       # TODO: np.array([book_embedding(tokenize(keywords[b]), glove) for b in seen])
reader_mean = None           # TODO: seen_embeddings.mean(axis=0)
reader_embedding = None      # TODO: reader_mean / np.linalg.norm(reader_mean)   (L2-normalize)
reader_is_unit_length = None # TODO: bool - np.isclose(np.linalg.norm(reader_embedding), 1.0)

normalize_explanation = "TODO"  # 2-3 sentences: L2-normalizing puts every book/reader on the unit
                                # sphere so cosine compares DIRECTION (meaning), and a long keyword
                                # bag cannot win on vector length alone


## Exercise 3 - Register the semantic path and read the scoreboard honestly

Now wrap the embedding retriever as a first-class path. `SemanticEmbeddingRetrievalPath(keywords, glove)` takes the keyword corpus and the GloVe subset at construction (like `LexicalRetrievalPath`), so `fit(rows, catalog=catalog_ids)` keeps the exact protocol signature and merely precomputes each book's embedding; `retrieve` builds the reader embedding from `context["seen"]` and scores the catalog by cosine. Register it in a `PathRegistry` (it must own the name `semantic-v1`), then score it on `val` beside lexical, the item-item CF and matrix-factorization collaborative paths, and the random floor.

Read the numbers **honestly**. Semantic should land around hit@10 ~0.102: clearly above the floor (~0.012, about 8.5x), but **below** lexical (~0.158) and far below item-item CF (~0.252) and MF (~0.276). This is a content path that clears the floor and loses to lexical here -- *not* a win. In three or four sentences, state that ordering plainly and explain **why** semantic trails lexical on this data: the latent-taste topics are arbitrary slices of a general word pool, so GloVe's meaning-geometry is uncorrelated-by-construction with the taste structure (only the genre banks help); a real catalog's prose would differ. Note too that precision@k and NDCG@k (the Unit 6 ranking metrics) rank the paths in the same order -- a weak path is weak on every lens.

In [ ]:
# Fit the paths once; later exercises REUSE these fitted objects. The MF fit is the only slow one
# (~15 s). Keep this to five fits.
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)       # collaborative filtering
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)   # matrix factorization
floor = RandomRetrievalPath(catalog_ids, seed=SEED)

# Register the semantic path and confirm it owns semantic-v1.
registry = PathRegistry()
registry.register(semantic)
artifact_name = None        # TODO: semantic.artifact_name()   -- 'semantic-v1'
registered_as_semantic = None  # TODO: bool - artifact_name == 'semantic-v1' and 'semantic' in registry


def val_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )


semantic_board = None   # TODO: val_hit(semantic)   -- hit ~0.102, precision ~0.011, ndcg ~0.033
lexical_hit = None      # TODO: val_hit(lexical).hit_rate_at_k        (~0.158)
cf_hit = None           # TODO: val_hit(item_item).hit_rate_at_k      (~0.252)
mf_hit = None           # TODO: val_hit(mf).hit_rate_at_k             (~0.276)
floor_hit = None        # TODO: val_hit(floor).hit_rate_at_k          (~0.012)
semantic_hit = None     # TODO: semantic_board.hit_rate_at_k          (~0.102)

clears_floor = None         # TODO: bool - semantic_hit > 5 * floor_hit
below_lexical = None        # TODO: bool - semantic_hit < lexical_hit
below_collaborative = None  # TODO: bool - lexical_hit < cf_hit and lexical_hit < mf_hit
honest_reading = "TODO"  # 3-4 sentences: semantic clears the floor (~8.5x) but is BELOW lexical and
                         # far below CF/MF -- NOT a win; it trails here because the latent topics are
                         # arbitrary word-pool slices (GloVe meaning is uncorrelated with taste by
                         # construction; only genre banks help) -- a real corpus would differ

## Exercise 4 - Where semantic earns its place: complementarity

A path that loses to lexical head-to-head can still be worth keeping -- if it is *wrong about different books*. That is exactly semantic's case. For every eligible warm reader, retrieve a top-10 from **lexical** and a top-10 from **semantic** and measure the **overlap** (the fraction of the 10 slots the two lists share); average it across readers. You should find the mean overlap is only about **0.22** -- the two paths agree on roughly two books in ten and surface eight different ones.

Then make it concrete for one reader: take a book that **semantic** ranks into its top-10 but **lexical** leaves out, and **count how many surface tokens it actually shares** with the reader's seen history. You will find the count is **not zero** -- lexical *could* have matched on those shared tokens too. The two paths disagree on **ranking**, not on vocabulary: lexical (BM25) ranks by weighted, saturated counts of matched terms, while semantic ranks by the cosine of the pooled GloVe vectors, so a book with a few thematically-central shared words can rank high for semantic yet fall outside lexical's top-10. In two or three sentences, explain why a low overlap (not a high hit@10) is what makes semantic a promising ingredient for a blend (the Challenge measures this directly).

In [ ]:
# Mean lexical-vs-semantic top-10 overlap across eligible warm readers.
overlaps = []
for rid in sorted(val_pos_by):
    seen_r = seen_by[rid]
    relevant = val_pos_by[rid] - seen_r
    if rid in cold_readers or not relevant:
        continue
    sem_ids = None   # TODO: [c.item_id for c in rank(semantic.retrieve(rid, {"seen": seen_r}, K), n=K, exclude=seen_r)]
    lex_ids = None   # TODO: [c.item_id for c in rank(lexical.retrieve(rid, {"seen": seen_r}, K), n=K, exclude=seen_r)]
    if sem_ids and lex_ids:
        overlaps.append(len(set(sem_ids) & set(lex_ids)) / K)   # TODO: fill sem_ids/lex_ids above

mean_overlap = None     # TODO: float(np.mean(overlaps))   -- about 0.22
overlap_is_low = None   # TODO: bool - 0.0 < mean_overlap < 0.4  (mostly DIFFERENT books)

# One reader, concretely: a book semantic ranks in but lexical drops.
example_reader = sorted(r for r in val_pos_by if r not in cold_readers and (val_pos_by[r] - seen_by[r]))[0]
ex_seen = seen_by[example_reader]
ex_sem = [c.item_id for c in rank(semantic.retrieve(example_reader, {"seen": ex_seen}, K), n=K, exclude=ex_seen)]
ex_lex = [c.item_id for c in rank(lexical.retrieve(example_reader, {"seen": ex_seen}, K), n=K, exclude=ex_seen)]
semantic_only = None    # TODO: sorted(set(ex_sem) - set(ex_lex))   -- books semantic ranks in, lexical does not
semantic_finds_extra = None   # TODO: bool - len(semantic_only) >= 1
example_book = None           # TODO: semantic_only[0]
example_keyword_bag = None    # TODO: keywords[example_book]

# Does this semantic-ranked-higher book really share NO tokens with the reader's history? COUNT them.
seen_tokens = set()           # TODO: build the union of tokenize(keywords[b]) over b in ex_seen:
#   for b in ex_seen:
#       seen_tokens |= set(tokenize(keywords[b]))
shared_tokens = None          # TODO: sorted(set(tokenize(keywords[example_book])) & seen_tokens)
shares_tokens = None          # TODO: bool - len(shared_tokens) > 0   (NONZERO: the disagreement is about RANKING)

complementarity_explanation = "TODO"  # 2-3 sentences: a LOW overlap (not a high hit@10) is what makes
                                      # semantic useful -- it RANKS books differently from lexical even when
                                      # they SHARE tokens (BM25 by weighted term counts, semantic by cosine of
                                      # pooled vectors), so it is wrong about DIFFERENT books and a blend can gain


## Exercise 5 - Challenge: max-similarity pooling vs the mean

Mean pooling blurs a reader's whole history into **one** centroid vector, which can wash out a reader with several distinct interests. An alternative, **max-similarity-over-seen**, scores each candidate book by its single *best* cosine to **any** one of the reader's seen books -- so a candidate that strongly matches even one past favourite ranks high. Measured on `val`, max-sim pooling scores hit@10 ~0.146, noticeably **above** mean pooling's ~0.102 (a documented alternative, not the shipped default).

Using the fitted path's precomputed book-embedding matrix (`semantic.artifact()["embeddings"]`, with `["item_ids"]` as the row order -- every row is already unit-length), build the **max-sim** top-10 for one warm reader and compare it to the mean-pooled top-10 you get from `semantic.retrieve`. They should differ. In three or four sentences, explain why taking a max over the seen books (rather than a mean) can rank better for a multi-interest reader, and name one cost of max-sim (it ignores how *consistent* a candidate is across the whole history).

In [ ]:
art = semantic.artifact()
book_matrix = np.array(art["embeddings"])     # (n_books, 100); each row L2-normalized (or zero)
item_ids = list(art["item_ids"])              # the row order of book_matrix
row_of = {iid: i for i, iid in enumerate(item_ids)}

stretch_reader = sorted(r for r in seen_by if r not in cold_readers)[0]
stretch_seen = seen_by[stretch_reader]
seen_rows = [row_of[b] for b in stretch_seen if b in row_of]

# max-sim-over-seen: each book's score is its BEST cosine to any one seen book.
sim_to_seen = None     # TODO: book_matrix @ book_matrix[seen_rows].T   -- (n_books, n_seen)
max_sim = None         # TODO: sim_to_seen.max(axis=1)                   -- best match per book
order = None           # TODO: np.argsort(-max_sim)
maxsim_top = None      # TODO: [item_ids[j] for j in order if item_ids[j] not in stretch_seen][:K]

# mean pooling (the shipped default) for the SAME reader, via the path.
mean_top = [c.item_id for c in rank(
    semantic.retrieve(stretch_reader, {"seen": stretch_seen}, K), n=K, exclude=stretch_seen
)]

poolings_differ = None   # TODO: bool - set(maxsim_top) != set(mean_top)
# Measured on val (seed 0, k=10, 500 readers): mean pooling ~0.102, max-sim-over-seen ~0.146.
maxsim_explanation = "TODO"  # 3-4 sentences: a MAX over seen books keeps a candidate that strongly
                             # matches even ONE past favourite (good for multi-interest readers), where
                             # the MEAN blurs tastes into one centroid; a cost of max-sim is ignoring
                             # consistency across the whole history

## Exercise 6 - Challenge: add semantic to the Unit 6 blend

This is the quantitative complementarity test. Reuse the Unit 6 **pinned** blend (per-path pool 30; weights `{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}`) and measure its hit@10 and catalog coverage on `val`. Then add the semantic path at a modest weight (0.5) and measure again. Even though semantic *loses* to lexical on its own (Exercise 3), it should lift **both** numbers here -- hit@10 from ~0.306 to ~0.316 and coverage from ~0.333 to ~0.353 -- because it pulls in meaning-related books the other paths miss.

Fit the one remaining path (popularity), reuse the Exercise-3 fits, and compute the two blended scoreboards. In two or three **honest** sentences: this is the complementarity claim made quantitative (a path below lexical still helps the blend), but it is a result at **this pinned config** -- the semantic weight is chosen in advance, not swept on `val`, so the lift is not a validation-sweep maximum reported as expected performance.

In [ ]:
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)   # the one path not yet fitted

WEIGHTS_U6 = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}   # the Unit 6 pinned weights
POOL = 30                                                                           # the Unit 6 pinned pool
paths_without = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}
paths_with = dict(paths_without, semantic=semantic)
weights_with = dict(WEIGHTS_U6, semantic=0.5)   # add semantic at a modest, pre-chosen weight


def blended(path_map, weight_map):
    return run_blended_scoreboard(
        path_map, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
        weights=weight_map, cold_readers=cold_readers,
    )


without_semantic = None   # TODO: blended(paths_without, WEIGHTS_U6)        (hit ~0.306 / cov ~0.333)
with_semantic = None      # TODO: blended(paths_with, weights_with)         (hit ~0.316 / cov ~0.353)

hit_rises = None       # TODO: bool - with_semantic.hit_rate_at_k > without_semantic.hit_rate_at_k
coverage_rises = None  # TODO: bool - with_semantic.catalog_coverage > without_semantic.catalog_coverage
blend_reading = "TODO"  # 2-3 HONEST sentences: adding semantic (which LOSES to lexical alone) lifts BOTH
                        # hit AND coverage -- complementarity made quantitative; but this is at the PINNED
                        # config, the semantic weight pre-chosen not swept, so it is not a val-sweep maximum

## Exercise 7 - Challenge: the dot-product retriever that connects three units

Semantic retrieval, Unit 5's matrix factorization, and Unit 8's coming two-tower are the **same retriever** wearing different clothes: each represents a reader and a book as vectors in one shared space and scores them by a **dot product** (a cosine, once the vectors are unit-length). The only thing that changes is *where the vectors come from* -- GloVe's are **pretrained** (fixed, from Wikipedia text), MF's are **learned** from the interaction log, and the two-tower's are **learned by a neural network**.

Make the equivalence concrete. Rebuild one reader's semantic score **by hand** as a dot product -- the L2-normalized mean of their seen books' embedding rows, dotted against the book matrix -- and confirm your top book and its score match what `semantic.retrieve` returns. Then look at the MF factors (`mf.artifact()`): its reader matrix `P` and item matrix `Q` live in the same factor dimension, and MF scores reader `r` against book `i` as `P[r] . Q[i]` -- the identical operation on *learned* vectors. In three or four sentences, describe the bridge: pretrained (GloVe, here), learned (MF, Unit 5), and learned-neural (two-tower, Unit 8) are all dot-product retrievers over embeddings; only the source of the vectors differs.

In [ ]:
# Reuse Exercise 5's book_matrix / item_ids / row_of (the path's own precomputed embeddings).
bridge_reader = sorted(r for r in seen_by if r not in cold_readers)[0]
bridge_seen = seen_by[bridge_reader]
bridge_rows = [row_of[b] for b in bridge_seen if b in row_of]

# By hand: reader vector = L2-normalized mean of seen rows; score every book by a dot product.
reader_vec = None     # TODO: book_matrix[bridge_rows].mean(axis=0)
reader_vec_unit = None  # TODO: reader_vec / np.linalg.norm(reader_vec)
hand_scores = None    # TODO: book_matrix @ reader_vec_unit   -- cosine of every book vs the reader

# The top unseen book by the hand dot product.
hand_order = None     # TODO: np.argsort(-hand_scores)
hand_top_id = None    # TODO: next(item_ids[j] for j in hand_order if item_ids[j] not in bridge_seen)
hand_top_score = None # TODO: float(hand_scores[row_of[hand_top_id]])

# What the path returns for the same reader.
path_top = rank(semantic.retrieve(bridge_reader, {"seen": bridge_seen}, K), n=K, exclude=bridge_seen)[0]
matches_path = None   # TODO: bool - hand_top_id == path_top.item_id and np.isclose(hand_top_score, path_top.score, atol=1e-5)

# MF does the SAME dot product on LEARNED factors: score(r, i) = P[r] . Q[i].
mf_art = mf.artifact()
P = np.array(mf_art["P"])   # reader factors (readers x n_factors)
Q = np.array(mf_art["Q"])   # item   factors (items   x n_factors)
same_factor_space = None    # TODO: bool - P.shape[1] == Q.shape[1]   (reader and item vectors share one space)

bridge_explanation = "TODO"  # 3-4 sentences: semantic (pretrained GloVe), MF (learned factors, Unit 5),
                             # and the two-tower (learned-neural, Unit 8) are ALL dot-product retrievers
                             # over reader/book embeddings -- only the SOURCE of the vectors differs